## Stage 0 — World and LLM Setup

In [1]:
# ── World ──────────────────────────────────────────────────────────────────────
# Uses the PR2-apartment fixture that ships with the existing llmr package.
# Provides: world (SDT), pr2 (robot annotation), context (PyCRAM Context)
from llmr.world_setup import load_pr2_apartment_world

world, pr2, context = load_pr2_apartment_world()

print("Bodies in world:")
for body in world.bodies:
    print(f"  {body.name}")

# ── Instruction ────────────────────────────────────────────────────────────────
INSTRUCTION = "pick up the milk from the table"
print(f"\nInstruction: {INSTRUCTION!r}")

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


Bodies in world:
  pr2/base_footprint
  pr2/base_link
  pr2/base_bellow_link
  pr2/base_laser_link
  pr2/fl_caster_rotation_link
  pr2/fl_caster_l_wheel_link
  pr2/fl_caster_r_wheel_link
  pr2/fr_caster_rotation_link
  pr2/fr_caster_l_wheel_link
  pr2/fr_caster_r_wheel_link
  pr2/bl_caster_rotation_link
  pr2/bl_caster_l_wheel_link
  pr2/bl_caster_r_wheel_link
  pr2/br_caster_rotation_link
  pr2/br_caster_l_wheel_link
  pr2/br_caster_r_wheel_link
  pr2/torso_lift_link
  pr2/l_torso_lift_side_plate_link
  pr2/r_torso_lift_side_plate_link
  pr2/torso_lift_motor_screw_link
  pr2/imu_link
  pr2/head_pan_link
  pr2/head_tilt_link
  pr2/head_plate_frame
  pr2/sensor_mount_link
  pr2/high_def_frame
  pr2/high_def_optical_frame
  pr2/double_stereo_link
  pr2/wide_stereo_link
  pr2/wide_stereo_optical_frame
  pr2/wide_stereo_l_stereo_camera_frame
  pr2/wide_stereo_l_stereo_camera_optical_frame
  pr2/wide_stereo_r_stereo_camera_frame
  pr2/wide_stereo_r_stereo_camera_optical_frame
  pr2/narrow_s

In [2]:
# ── LLM ───────────────────────────────────────────────────────────────────────
# Swap provider/model freely — everything downstream accepts any BaseChatModel.
# Set your OPENAI_API_KEY env var before running, or switch to OLLAMA for local.
import os
from llm_reasoner.workflows.llm_config import make_llm, LLMProvider

os.environ["OPENAI_API_KEY"] = "REDACTED_API_KEY"   # uncomment if not already set

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
print(f"LLM ready: {llm.model_name}")

LLM ready: gpt-4o-mini


## Stage 1 — Discover Action Classes

`_discover_action_classes()` scans `pycram.robot_plans.actions` and returns
every concrete class whose name ends with `"Action"`. No manual registry needed.

In [3]:
from llm_reasoner.workflows.slot_filler import _discover_action_classes

action_registry = _discover_action_classes()

print(f"Discovered {len(action_registry)} action classes:\n")
for name, cls in sorted(action_registry.items()):
    print(f"  {name:30s}  →  {cls.__module__}.{cls.__qualname__}")

# Expected output:
# Discovered N action classes:
#   MoveTorsoAction       →  pycram.robot_plans.actions.core.MoveTorsoAction
#   NavigateAction        →  pycram.robot_plans.actions.core.NavigateAction
#   ParkArmsAction        →  pycram.robot_plans.actions.core.ParkArmsAction
#   PickUpAction          →  pycram.robot_plans.actions.core.PickUpAction
#   PlaceAction           →  pycram.robot_plans.actions.core.PlaceAction
#   ...

Discovered 24 action classes:

  CarryAction                     →  pycram.robot_plans.actions.core.robot_body.CarryAction
  CloseAction                     →  pycram.robot_plans.actions.core.container.CloseAction
  CuttingAction                   →  pycram.robot_plans.actions.composite.tool_based.CuttingAction
  DetectAction                    →  pycram.robot_plans.actions.core.misc.DetectAction
  EfficientTransportAction        →  pycram.robot_plans.actions.composite.transporting.EfficientTransportAction
  FaceAtAction                    →  pycram.robot_plans.actions.composite.facing.FaceAtAction
  FollowToolCenterPointPathAction  →  pycram.robot_plans.actions.core.robot_body.FollowToolCenterPointPathAction
  GraspingAction                  →  pycram.robot_plans.actions.core.pick_up.GraspingAction
  LookAtAction                    →  pycram.robot_plans.actions.core.navigation.LookAtAction
  MixingAction                    →  pycram.robot_plans.actions.composite.tool_based.MixingActio

## Stage 2 — LLM Classifies Action Type

The LLM receives the full registry list and the NL instruction.
It returns `ActionClassification` — an exact Python class name + reasoning.

In [4]:
from llm_reasoner.workflows.schemas.action_slot import ActionClassification
from llm_reasoner.workflows.slot_filler import _CLASSIFIER_SYSTEM_PROMPT

# ── Build the prompt the LLM will receive ─────────────────────────────────────
class_list = "\n".join(f"  - {name}" for name in sorted(action_registry))
system_prompt = _CLASSIFIER_SYSTEM_PROMPT.format(action_classes=class_list)

print("=== System prompt sent to LLM ===")
print(system_prompt)
print(f"\n=== User message ===")
print(f"Instruction: {INSTRUCTION!r}")

=== System prompt sent to LLM ===
You are a robot action classifier.

Given a natural-language instruction, identify which robot action class it
corresponds to from the list of available action classes below.

Available action classes:
  - CarryAction
  - CloseAction
  - CuttingAction
  - DetectAction
  - EfficientTransportAction
  - FaceAtAction
  - FollowToolCenterPointPathAction
  - GraspingAction
  - LookAtAction
  - MixingAction
  - MoveAndPickUpAction
  - MoveAndPlaceAction
  - MoveTorsoAction
  - NavigateAction
  - OpenAction
  - ParkArmsAction
  - PickAndPlaceAction
  - PickUpAction
  - PlaceAction
  - PouringAction
  - ReachAction
  - SearchAction
  - SetGripperAction
  - TransportAction

Return the EXACT Python class name (e.g. "PickUpAction" not "pick up action").
Return structured JSON.


=== User message ===
Instruction: 'pick up the milk from the table'


In [5]:
# ── Actual LLM call ───────────────────────────────────────────────────────────
structured_llm = llm.with_structured_output(ActionClassification)

classification: ActionClassification = structured_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": INSTRUCTION},
])

print("=== LLM response ===")
print(f"  action_type : {classification.action_type}")
print(f"  confidence  : {classification.confidence}")
print(f"  reasoning   : {classification.reasoning}")

# ── Resolve to actual class ───────────────────────────────────────────────────
action_cls = action_registry.get(classification.action_type)
print(f"\nResolved class: {action_cls}")

assert action_cls is not None, f"Unknown action type: {classification.action_type!r}"

# Expected output:
# === LLM response ===
#   action_type : PickUpAction
#   confidence  : 0.98
#   reasoning   : "'pick up' maps directly to PickUpAction. No other action class fits."
#
# Resolved class: <class 'pycram.robot_plans.actions.core.PickUpAction'>

=== LLM response ===
  action_type : PickUpAction
  confidence  : 0.95
  reasoning   : The instruction specifies to 'pick up' an item (milk) from a location (the table), which directly corresponds to the PickUpAction class.

Resolved class: <class 'pycram.robot_plans.actions.core.pick_up.PickUpAction'>


## Stage 3 — Build Fully-Underspecified Match

`_fully_underspecified()` introspects the action class's dataclass fields
and builds a `Match` with every field set to `...` (Ellipsis = free slot).

In [6]:
import dataclasses
from llm_reasoner.nl_factory import _fully_underspecified, _get_settable_fields

# ── Inspect what fields will be free ──────────────────────────────────────────
fields = _get_settable_fields(action_cls)
print(f"Settable fields on {action_cls.__name__}:")
for f in fields:
    type_hint = action_cls.__dataclass_fields__[f].type if dataclasses.is_dataclass(action_cls) else "?"
    print(f"  {f:30s}  type: {type_hint}")

# ── Build the Match ────────────────────────────────────────────────────────────
match = _fully_underspecified(action_cls)

print(f"\nMatch object : {match}")
print(f"Match type   : {match.type}")
print(f"Matches with variables ({len(list(match.matches_with_variables))}):")
for attr in match.matches_with_variables:
    print(f"  field={attr.name_from_variable_access_path:25s}  "
          f"value={attr.assigned_variable._value_!r:10}  "
          f"type={attr.assigned_variable._type_}")

# Expected output:
# Settable fields on PickUpAction:
#   object_designator              type: ObjectDesignatorDescription
#   arm                            type: Arms
#   grasp_description              type: GraspDescription
#
# Matches with variables (3):
#   field=object_designator    value=Ellipsis   type=ObjectDesignatorDescription
#   field=arm                  value=Ellipsis   type=Arms
#   field=grasp_description    value=Ellipsis   type=GraspDescription

Settable fields on PickUpAction:
  plan_node                       type: Optional[PlanNode]
  object_designator               type: Body
  arm                             type: Arms
  grasp_description               type: GraspDescription

Match object : Match(PickUpAction)
Match type   : <class 'pycram.robot_plans.actions.core.pick_up.PickUpAction'>
Matches with variables (4):
  field=PickUpAction.plan_node     value=Ellipsis    type=<class 'pycram.plans.plan_node.PlanNode'>
  field=PickUpAction.object_designator  value=Ellipsis    type=<class 'semantic_digital_twin.world_description.world_entity.Body'>
  field=PickUpAction.arm           value=Ellipsis    type=<enum 'Arms'>
  field=PickUpAction.grasp_description  value=Ellipsis    type=<class 'pycram.datastructures.grasp.GraspDescription'>


## Stage 4 — Inspect Free vs Fixed Slots

This is what `LLMBackend._evaluate()` does internally to split the Match into
free slots (need LLM) and fixed slots (already provided by the user).
Here we run it manually to make the split visible.

In [7]:
free_slots  = []   # (field_name, field_type) — LLM must fill these
fixed_slots = {}   # field_name → value     — already known, pass to LLM as context

for attr_match in match.matches_with_variables:
    field_name = attr_match.name_from_variable_access_path
    value      = attr_match.assigned_variable._value_
    field_type = attr_match.assigned_variable._type_

    if isinstance(value, type(Ellipsis)):
        free_slots.append((field_name, field_type))
    else:
        fixed_slots[field_name] = value

print("FREE slots  (LLM will fill these):")
for name, typ in free_slots:
    print(f"  {name:30s}  type: {typ}")

print("\nFIXED slots (already known, passed as context):")
if fixed_slots:
    for name, val in fixed_slots.items():
        print(f"  {name:30s}  =  {val!r}")
else:
    print("  (none — user left everything open)")

# Expected output (fully-underspecified Match — nothing pre-fixed):
# FREE slots  (LLM will fill these):
#   object_designator              type: ObjectDesignatorDescription
#   arm                            type: Arms
#   grasp_description              type: GraspDescription
#
# FIXED slots (already known, passed as context):
#   (none — user left everything open)

FREE slots  (LLM will fill these):
  PickUpAction.plan_node          type: <class 'pycram.plans.plan_node.PlanNode'>
  PickUpAction.object_designator  type: <class 'semantic_digital_twin.world_description.world_entity.Body'>
  PickUpAction.arm                type: <enum 'Arms'>
  PickUpAction.grasp_description  type: <class 'pycram.datastructures.grasp.GraspDescription'>

FIXED slots (already known, passed as context):
  (none — user left everything open)


## Stage 5 — Serialize World for LLM Context

`serialize_world()` converts the SDT world into a plain-text representation
the LLM can reason over: body names, positions, bounding boxes, and semantic annotations.
This is the LLM's entire view of the physical world.

In [8]:
from llm_reasoner.world_serializer import serialize_world

world_context = serialize_world(world)
print(world_context)

# Expected output (will vary by world fixture):
#
# === Scene Objects ===
#   - milk_bottle_1   position=(1.20, 0.85, 0.72)  dims=(w=0.07, d=0.07, h=0.24)
#   - cereal_box_1    position=(1.20, 1.10, 0.72)  dims=(w=0.08, d=0.15, h=0.30)
#   - kitchen_table   position=(1.20, 0.95, 0.60)  dims=(w=0.80, d=1.20, h=0.60)
#   - fridge_1        position=(2.50, 0.30, 0.90)
#   - pr2_robot       position=(0.00, 0.00, 0.00)
#
# === Semantic Annotations ===
#   - milk_bottle_1: FoodItem, Graspable
#   - cereal_box_1:  FoodItem, Graspable
#   - kitchen_table: SupportSurface, Furniture
#   - fridge_1:      Furniture, StorageUnit

=== Scene Objects ===
  - map  position=(0.00, 0.00, 0.00)
  - odom_combined  position=(0.00, 0.00, 0.00)
  - milk.stl  position=(2.37, 2.00, 1.05)  dims=(w=0.06, d=0.07, h=0.19)
  - breakfast_cereal.stl  position=(2.37, 1.80, 1.05)  dims=(w=0.15, d=0.06, h=0.22)

=== Semantic Annotations ===
  - milk.stl: Milk
  - breakfast_cereal.stl: Cereal


## Stage 6 — LLM Slot Filling (Core Reasoning)

`run_slot_filler()` is the heart of `LLMBackend._evaluate()`.
The LLM receives the full world context, instruction, action type,
and free slot names/types — and reasons to produce a concrete value for each.

In [9]:
from llm_reasoner.workflows.slot_filler import _build_slot_filler_message, _SLOT_FILLER_SYSTEM_PROMPT

# ── Inspect the exact user message the LLM will receive ───────────────────────
user_message = _build_slot_filler_message(
    instruction   = INSTRUCTION,
    action_type   = action_cls.__name__,
    free_slots    = free_slots,
    fixed_slots   = fixed_slots,
    world_context = world_context,
)

print("=== System prompt ===")
print(_SLOT_FILLER_SYSTEM_PROMPT)
print("\n=== User message ===")
print(user_message)

# Expected user message:
# Instruction: 'pick up the milk from the table'
# Action type: PickUpAction
#
# Free slots to fill:
#   - object_designator  (type: ObjectDesignatorDescription)
#   - arm                (type: Arms)
#   - grasp_description  (type: GraspDescription)
#
# Already-fixed slots (honour these, do not change):
#   (none)
#
# === Scene Objects ===
#   - milk_bottle_1   position=(1.20, 0.85, 0.72) ...
#   ...

=== System prompt ===
You are a robot action parameter resolver with strong spatial and physical reasoning.

You receive:
  1. A natural-language instruction from a human operator.
  2. The target robot action type and all its free (unfilled) parameter slots.
  3. Already-fixed slot values (do not change these).
  4. The full current world state: objects, positions, semantic annotations.

Your task: for every FREE slot, reason carefully and produce the most physically
sensible concrete value.

Guidelines per slot type:
  ENTITY slots (object_designator, target, surface, etc.):
    - Identify which world object the instruction refers to.
    - Use name matching, semantic type, spatial context, and attributes.
    - If multiple candidates exist, pick the most contextually salient one
      (closest to robot, most recently mentioned, best matching description).
    - Return the exact world body NAME as the value (it will be looked up).

  ENUM / PARAMETER slots (arm, grasp_type, approach_

In [10]:
from llm_reasoner.workflows.slot_filler import run_slot_filler

# ── Actual LLM reasoning call ─────────────────────────────────────────────────
resolved = run_slot_filler(
    instruction   = INSTRUCTION,
    action_type   = action_cls.__name__,
    free_slots    = free_slots,
    fixed_slots   = fixed_slots,
    world_context = world_context,
    llm           = llm,
)

print("=== Resolved slot values from LLM ===\n")
if resolved:
    for field_name, value in resolved.items():
        print(f"  {field_name:30s}  →  {value!r}")
else:
    print("  LLM returned None — check API key / model availability")

# Expected output:
# === Resolved slot values from LLM ===
#
#   object_designator              →  'milk_bottle_1'
#   arm                            →  'RIGHT'
#   grasp_description              →  {'approach_direction': 'FRONT',
#                                      'vertical_alignment': 'TOP',
#                                      'rotate_gripper': False}

assert resolved is not None, "Slot filler returned None — LLM call failed"

=== Resolved slot values from LLM ===

  PickUpAction.plan_node          →  'PickUpAction.plan_node'
  PickUpAction.object_designator  →  'milk.stl'
  PickUpAction.arm                →  'RIGHT'
  PickUpAction.grasp_description  →  'PickUpAction.grasp_description'


dict_items([('PickUpAction.plan_node', 'PickUpAction.plan_node'), ('PickUpAction.object_designator', 'milk.stl'), ('PickUpAction.arm', 'RIGHT'), ('PickUpAction.grasp_description', 'PickUpAction.grasp_description')])

## Stage 7 — Entity Name → World Body Lookup

The LLM returns entity slots as **body name strings** (e.g. `"milk_bottle_1"`).
Before assigning to the Match variable, we resolve them to actual `Body` objects.
This is the only deterministic step — pure validation, no reasoning.

In [13]:
def resolve_entity(name: str, world) -> object:
    """
    Look up a body by name in the world.
    Raises ValueError if the LLM hallucinated a name that doesn't exist.
    This is the ONLY deterministic step in the whole pipeline.
    """
    bodies = world.get_bodies_by_name(name)
    if not bodies:
        raise ValueError(
            f"LLM resolved entity to {name!r} but no body with that name "
            f"exists in the world. Available: {[str(b.name) for b in world.bodies]}"
        )
    return bodies[0]


# ── Identify entity slots (those whose LLM value is a string body name) ───────
# In a full implementation, slot type annotations tell us which are entity slots.
# For this notebook we use a simple heuristic: string values against known body names.
known_body_names = {str(b.name) for b in world.bodies}

resolved_typed = {}
for field_name, value in resolved.items():
    if isinstance(value, str) and value in known_body_names:
        body = resolve_entity(value, world)
        resolved_typed[field_name] = body
        print(f"  Entity resolved: {field_name} = {value!r}  →  {body}")
    else:
        resolved_typed[field_name] = value
        print(f"  Parameter kept:  {field_name} = {value!r}")

# Expected output:
#   Entity resolved: object_designator = 'milk_bottle_1'  →  Body(milk_bottle_1)
#   Parameter kept:  arm               = 'RIGHT'
#   Parameter kept:  grasp_description = {'approach_direction': 'FRONT', ...}

  Parameter kept:  PickUpAction.plan_node = 'PickUpAction.plan_node'
  Entity resolved: PickUpAction.object_designator = 'milk.stl'  →  Body(name=PrefixedName('None/milk.stl'), id=UUID('e430be4b-d3b2-49be-bbde-2478d7df66ab'), index=219)
  Parameter kept:  PickUpAction.arm = 'RIGHT'
  Parameter kept:  PickUpAction.grasp_description = 'PickUpAction.grasp_description'


## Stage 8 — Write Values Back into Match → Construct Instance

This mirrors the exact pattern used by `ProbabilisticBackend` and
`EntityQueryLanguageBackend._generate_raw_results()` in `krrood`:

```python
expression._get_mapped_variable_by_name(field_name)._value_ = value
expression._update_kwargs_from_literal_values()
yield expression.construct_instance()
```

In [ ]:
print("=== Before assignment — Match variables ===")
for attr in match.matches_with_variables:
    name = attr.name_from_variable_access_path
    print(f"  {name:30s}  _value_ = {attr.assigned_variable._value_!r}")

# ── Assign resolved values into the Match variable graph ──────────────────────
print("\n=== Assigning LLM-resolved values ===")
for field_name, value in resolved_typed.items():
    mapped_var = match._get_mapped_variable_by_name(field_name)
    if mapped_var is not None:
        mapped_var._value_ = value
        print(f"  {field_name:30s}  ←  {value!r}")
    else:
        print(f"  WARNING: no mapped variable found for {field_name!r}")

# ── Sync kwargs and construct the concrete action instance ─────────────────────
match._update_kwargs_from_literal_values()
action_instance = match.construct_instance()

print(f"\n=== Constructed action instance ===")
print(f"  Type : {type(action_instance).__name__}")
print(f"  Value: {action_instance}")

# Expected output:
# === Before assignment — Match variables ===
#   object_designator              _value_ = Ellipsis
#   arm                            _value_ = Ellipsis
#   grasp_description              _value_ = Ellipsis
#
# === Assigning LLM-resolved values ===
#   object_designator              ←  Body(milk_bottle_1)
#   arm                            ←  'RIGHT'
#   grasp_description              ←  {'approach_direction': 'FRONT', ...}
#
# === Constructed action instance ===
#   Type : PickUpAction
#   Value: PickUpAction(object_designator=Body(milk_bottle_1), arm='RIGHT', ...)

## Stage 9 — Power-User API: Manual Backend + `execute_single()`

Instead of the fully-automatic `nl_plan()`, the power user can pre-fix
some slots (e.g. force a specific arm) and let the LLM fill only the rest.

In [ ]:
from krrood.entity_query_language.query.match import Match
from llm_reasoner.backend import LLMBackend
from pycram.plans.factories import execute_single

# Power user pre-fixes arm=LEFT — LLM only has to fill object_designator + grasp_description
from pycram.datastructures.enums import Arms

power_match = Match(action_cls)(
    object_designator = ...,        # free — LLM will ground "milk from table"
    arm               = Arms.LEFT,  # fixed — user forces left arm
    grasp_description = ...,        # free — LLM will reason about grasp
)

print("Power-user Match:")
for attr in power_match.matches_with_variables:
    name  = attr.name_from_variable_access_path
    value = attr.assigned_variable._value_
    status = "FREE (LLM fills)" if isinstance(value, type(Ellipsis)) else f"FIXED = {value!r}"
    print(f"  {name:30s}  {status}")

# ── Wire LLMBackend and wrap in plan node ─────────────────────────────────────
context.query_backend = LLMBackend(
    instruction = INSTRUCTION,
    llm         = llm,
    world       = world,
)

plan_node = execute_single(power_match, context)
print(f"\nPlan node created: {plan_node}")
print(f"Backend on context: {type(context.query_backend).__name__}")

# Expected output:
# Power-user Match:
#   object_designator              FREE (LLM fills)
#   arm                            FIXED = Arms.LEFT
#   grasp_description              FREE (LLM fills)
#
# Plan node created: <UnderspecifiedNode ...>
# Backend on context: LLMBackend

## Stage 10 — Simple User API: `nl_plan()` end-to-end

`nl_plan()` runs all previous stages automatically.
The user provides only the NL instruction — nothing else.

In [ ]:
from llm_reasoner import nl_plan

# ── One call — all 9 stages happen automatically inside ───────────────────────
plan_node = nl_plan(
    instruction = INSTRUCTION,
    context     = context,
    llm         = llm,
)

print(f"Plan node  : {plan_node}")
print(f"Backend    : {type(context.query_backend).__name__}")
print(f"Instruction: {context.query_backend.instruction!r}")

# Expected output:
# Plan node  : <UnderspecifiedNode: PickUpAction>
# Backend    : LLMBackend
# Instruction: 'pick up the milk from the table'

## Stage 11 — Execute

`plan.perform()` triggers `UnderspecifiedNode._perform()` which calls
`context.query_backend.evaluate(match)` → `LLMBackend._evaluate()` → robot motion.

> **Note:** Run this cell only in a live simulation environment (Multiverse / Bullet).
> Comment out `plan_node.perform()` and use `with simulated_robot:` if needed.

In [ ]:
# Uncomment `with simulated_robot:` if your environment requires it.
# from pycram.process_module import simulated_robot

# with simulated_robot:
#     plan_node.perform()

plan_node.perform()

# Expected console output:
# [INFO] LLMBackend._evaluate: resolving PickUpAction
# [INFO]   → object_designator : Body(milk_bottle_1)
# [INFO]   → arm               : Arms.RIGHT
# [INFO]   → grasp_description : GraspDescription(FRONT, TOP, rotate=False)
# [INFO] PickUpAction.perform() — grasping milk_bottle_1 with RIGHT arm
# [INFO] Motion complete.

## Bonus — Multi-step: `nl_sequential()`

Tests `TaskDecomposer` + `nl_plan()` chained together for a compound instruction.

In [ ]:
from llm_reasoner.task_decomposer import TaskDecomposer

MULTI_INSTRUCTION = "go to the table, pick up the milk, and put it in the fridge"

# ── Stage A: Decompose ────────────────────────────────────────────────────────
decomposer = TaskDecomposer(llm=llm)
decomposed = decomposer.decompose(MULTI_INSTRUCTION)

print("=== Decomposed steps ===")
for i, step in enumerate(decomposed.steps):
    deps = decomposed.dependencies.get(i, [])
    dep_str = f"  (depends on steps {deps})" if deps else ""
    print(f"  [{i}] {step}{dep_str}")

# Expected output:
# === Decomposed steps ===
#   [0] navigate to the table
#   [1] pick up the milk from the table   (depends on steps [0])
#   [2] place the milk in the fridge      (depends on steps [1])

In [ ]:
from llm_reasoner import nl_sequential

# ── Stage B: Build one plan node per step ─────────────────────────────────────
plan_nodes = nl_sequential(
    instruction = MULTI_INSTRUCTION,
    context     = context,
    llm         = llm,
)

print(f"Built {len(plan_nodes)} plan nodes:\n")
for i, node in enumerate(plan_nodes):
    print(f"  [{i}] {node}  —  backend instruction: {context.query_backend.instruction!r}")

# ── Stage C: Execute sequentially ─────────────────────────────────────────────
print("\nExecuting...")
for i, node in enumerate(plan_nodes):
    print(f"\n--- Step {i} ---")
    node.perform()

# Expected output:
# Built 3 plan nodes:
#   [0] <UnderspecifiedNode: NavigateAction>  — backend instruction: 'navigate to the table'
#   [1] <UnderspecifiedNode: PickUpAction>    — backend instruction: 'pick up the milk from the table'
#   [2] <UnderspecifiedNode: PlaceAction>     — backend instruction: 'place the milk in the fridge'
#
# Executing...
# --- Step 0 ---
# [INFO] NavigateAction.perform() — navigating to kitchen_table
# --- Step 1 ---
# [INFO] PickUpAction.perform()   — grasping milk_bottle_1 with RIGHT arm
# --- Step 2 ---
# [INFO] PlaceAction.perform()    — placing milk_bottle_1 in fridge_1